# Week 2, Day 3: Clothes-store chatbot

A Gradio chat that keeps the conversation history and streams replies from Gemini.
The system prompt steers the assistant toward items on sale.

In [ ]:
import os
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr

load_dotenv(override=True)
api_key = os.getenv("GEMINI_API_KEY")
print("GEMINI_API_KEY found" if api_key else "GEMINI_API_KEY not set")

MODEL = "gemini-3.5-flash-lite"
gemini = OpenAI(base_url="https://generativelanguage.googleapis.com/v1beta/openai/", api_key=api_key)

STORE = """
You are a helpful assistant in a clothes store. You should try to gently encourage
the customer to try items that are on sale. Hats are 60% off, and most other items are 50% off.
For example, if the customer says 'I'm looking to buy a hat',
you could reply something like, 'Wonderful - we have lots of hats - including several that are part of our sales event.'
Encourage the customer to buy hats if they are unsure what to get.
If the customer asks for shoes, respond that shoes are not on sale today, but remind the customer to look at hats.
"""

## Streaming chat with history

Gradio passes the new message and the prior turns. Each call still sends the full history, because the model is stateless.
If the latest message mentions belts, the system prompt adds that the store does not sell them.

In [ ]:
def chat(message, history):
    system = STORE
    if "belt" in message.lower():
        system += " The store does not sell belts; if you are asked for belts, be sure to point out other items on sale."
    messages = [{"role": "system", "content": system}]
    messages += [{"role": turn["role"], "content": turn["content"]} for turn in history]
    messages.append({"role": "user", "content": message})
    stream = gemini.chat.completions.create(model=MODEL, messages=messages, stream=True)
    response = ""
    for chunk in stream:
        response += chunk.choices[0].delta.content or ""
        yield response


gr.ChatInterface(fn=chat, type="messages").launch()